# Practica de Programacion y Algoritmia Avanzada – GIA

El objetivo de esta actividad es poner en practica los conocimientos de programacion dinamica, para tal caso, se os propone implementar el algoritmo determinista Cocke–Kasami–Younger.

El algoritmo CKY determinista es un algoritmo utilizado para analizar la estructura sintactica de cadenas de texto, y determinar si pueden ser derivadas apartir de las reglas de una gramatica.

In [ ]:
# INPUT
# Gramar = {} # CNF grammar
# Word = []

# terminals = set()
# non_terminals = set()

In [11]:
# MAIN PROGRAM
def input_grammar(input_grammar):
    # Read grammar from input
    grammar = {}
    while True:
        line = input().strip()
        if line == "":
            break
        lhs, rhs = line.split("->")
        lhs = lhs.strip()
        rhs = rhs.strip().split("|")
        grammar[lhs] = [r.strip() for r in rhs]
    return grammar
    
def check_grammar(grammar):
    # Check if the grammar is in CNF
    for lhs, rhs_list in grammar.items():
        for rhs in rhs_list:
            if len(rhs) > 2 or (len(rhs) == 2 and not all(c.isupper() for c in rhs)):
                print(f"Grammar is not in CNF: {lhs} -> {rhs}")
                return False
    print("Grammar is in CNF")
    
def check_word(word, grammar):
    # Check if the word can be generated by the grammar
    def generate(lhs, word):
        if not word:
            return lhs == ""
        if lhs not in grammar:
            return False
        for rhs in grammar[lhs]:
            if len(rhs) == 1 and rhs[0] == word[0]:
                if generate(rhs[0], word[1:]):
                    return True
            elif len(rhs) == 2 and rhs[0] == word[0] and rhs[1] == word[1]:
                if generate(rhs[1], word[2:]):
                    return True
        return False
    
def test(word, grammar):
    # Test if the word can be generated by the grammar:
    # input_grammar(input_grammar)
    # grammar = input_grammar(input_grammar)
    if not check_grammar(grammar):
        return False
    else:
        return check_word(word, grammar)     


In [7]:
def cargar_gramatica(ruta_archivo):
    reglas = {}

    with open(ruta_archivo) as archivo:
        for linea in archivo:
            linea = linea.strip()
            if '->' in linea:
                

                lhs, rhs = linea.split('->')
                lhs = lhs.strip()
                alternativas = rhs.split('|')

                for alt in alternativas:
                    produccion = alt.strip().split()
                    if produccion:
                        reglas.setdefault(lhs, []).append(produccion)

    return reglas


In [9]:
gramatica= cargar_gramatica('c:/Users/veron/Documents/UNI/2º/Febrero/PAA/Practica/input.txt')


In [20]:
def es_cnf(gramatica):
    no_terminales = set(gramatica.keys())
    terminales = set()

    # Recorremos toda la gramática para ver qué símbolos podrían ser terminales
    for reglas in gramatica.values():
        for produccion in reglas:
            for simbolo in produccion:
                if simbolo not in no_terminales:
                    terminales.add(simbolo)

    # Ahora comprobamos si cada producción cumple las reglas de la CNF
    for lado_izq, reglas in gramatica.items():
        for produccion in reglas:
            if len(produccion) == 1:
                # Caso A -> a
                if produccion[0] not in terminales:
                    return False
            elif len(produccion) == 2:
                # Caso A -> B C
                if produccion[0] in terminales or produccion[1] in terminales:
                    return False
            else:
                # Producción con más de 2 símbolos ej A-> BCD
                return False

    return True


In [11]:
es_cnf(gramatica)

True

In [18]:
def cky(gramatica, palabra):
    n = len(palabra)
    tabla = [[set() for _ in range(n)] for _ in range(n)]

    # Crear un diccionario inverso: RHS -> LHS
    inversa = {}
    for lhs, producciones in gramatica.items():
        for rhs in producciones:
            clave = tuple(rhs)
            inversa.setdefault(clave, []).append(lhs)

    # Rellenar la diagonal (longitud 1): A -> a
    for i, simbolo in enumerate(palabra):
        if (simbolo,) in inversa:
            tabla[i][i].update(inversa[(simbolo,)])

    # Rellenar el resto de la tabla
    for longitud in range(2, n + 1):  # tamaño del fragmento
        for i in range(n - longitud + 1):
            j = i + longitud - 1
            for k in range(i, j):
                izquierda = tabla[i][k]
                derecha = tabla[k + 1][j]
                for B in izquierda:
                    for C in derecha:
                        if (B, C) in inversa:
                            tabla[i][j].update(inversa[(B, C)])

    # Verificamos si el símbolo inicial S genera toda la cadena
    return 'S' in tabla[0][n - 1]


In [21]:
Gramar1 = {
    'S': [['a'], ['X', 'A'], ['A', 'X'], ['b']],
    'A': [['R', 'B']],
    'B': [['A', 'X'], ['b'], ['a']],
    'X': [['a']],
    'R': [['X', 'B']]
}

Gramar2 = {
    'S': [['A','B'], ['C','D'], ['C','B'], ['S','S']],
    'A': [['B','C'], ['a']],
    'B': [['S','C'], ['b']],
    'C': [['D','D'], ['b']],
    'D': [['B','A']]
}

print(es_cnf(Gramar1))
print(es_cnf(Gramar2))
print(cky(Gramar1,['a', 'b', 'a', 'b']))
print(cky(Gramar1,['a', 'b', 'b', 'a']))


True
True
False
True


In [ ]:
# OUTPUT
# Boolean 

In [8]:
# TEST CASES

Gramar1 = {
    'S': [['a'], ['X', 'A'], ['A', 'X'], ['b']],
    'A': [['R', 'B']],
    'B': [['A', 'X'], ['b'], ['a']],
    'X': [['a']],
    'R': [['X', 'B']]
}

Gramar2 = {
    'S': [['A','B'], ['C','D'], ['C','B'], ['S','S']],
    'A': [['B','C'], ['a']],
    'B': [['S','C'], ['b']],
    'C': [['D','D'], ['b']],
    'D': [['B','A']]
}

In [15]:
# EXAMPLE 1 TEST CASE 1
Gramar = Gramar1
Word1 = ['a', 'b', 'a', 'b']

boolean1 = test(Word1, Gramar)
print(f"Word {Word1} can be generated by the grammar: {boolean1}")
#False

# EXAMPLE 2 TEST CASE 1
Word2 = ['a', 'b', 'b', 'a']
boolean2 = test(Word2, Gramar)
print(f"Word {Word2} can be generated by the grammar: {boolean2}")
# True


Grammar is in CNF
Word ['a', 'b', 'a', 'b'] can be generated by the grammar: False
Grammar is in CNF
Word ['a', 'b', 'b', 'a'] can be generated by the grammar: False


In [13]:
# EXAMPLE 2
Gramar = Gramar2
Word = ['a', 'b', 'b', 'a']

boolean = test(Word, Gramar)
print(f"Word {Word} can be generated by the grammar: {boolean}")

Grammar is in CNF
Word ['a', 'b', 'b', 'a'] can be generated by the grammar: False


In [ ]:
# EXTENSION 1
# Transformacion de cualquier gramatica CFG a CNF.

In [ ]:
# EXTENSION 2
# ampliar el algoritmo CKY a probabilıstico.